# Finwise Scribe — LoRA Fine-Tuning (Google Colab)

**Purpose:** Fine-tune a quantized Llama-3 8B model on a neuro-symbolic token language derived from OHLCV market data.

**Workflow:**
1. Install dependencies (Unsloth, TRL, PEFT)
2. Fetch 10 years of OHLCV data from Stooq and tokenize into a 10×10 Decile Matrix
3. Fine-tune Llama-3 8B with LoRA adapters
4. Evaluate the SLM against the LSTM baseline (RMSE)
5. Export the LoRA adapter to Google Drive

**Target Runtime:** Google Colab — T4 or A100 GPU

In [ ]:
# ==============================================================
# CELL 1: Environment Setup
# Installs Unsloth (optimized LoRA fine-tuning library) along with
# supporting packages for training, data ingestion, and evaluation.
# NOTE: Designed for Google Colab — run once per session.
# ==============================================================

import os

print("Installing dependencies — this may take 1–2 minutes...")
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes --quiet
!pip install pandas-datareader scikit-learn --quiet

# Disable Weights & Biases logging to prevent authentication prompts
# during non-tracked training runs.
os.environ["WANDB_DISABLED"] = "true"

print("Setup complete. WandB logging disabled.")

Kurulumlar yapılıyor... (1-2 dk)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.5/283.5 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 22.9 MB/s eta 0:00:00
Kurulum tamamlandı. WandB devre

## Step 1b — Training Configuration
Central configuration for all hyperparameters. Edit **only this cell** when experimenting with different settings before a training run.

In [ ]:
# ==============================================================
# CELL 1b: Training Configuration — SLM v2 (E3 Ablation Target)
#
# All hyperparameters are centralised here. The settings below
# correspond to the E3 ablation target in the roadmap:
#   Qwen2-1.5B | fixed semantic bins | 30-token context |
#   3000 steps | 10 tickers → target RMSE < 0.01389
#
# Edit ONLY this cell when running ablation experiments (E0–E3):
#   E1: keep everything, set context_window=30, max_seq=512, max_steps=300
#   E2: E1 settings + set use_fixed_bins=True, max_steps=300
#   E3: this full config (Qwen2, fixed bins, 30-token, 3000 steps, 10 tickers)
# ==============================================================

CFG = {
    # ── Data ──────────────────────────────────────────────────
    # E3 requires 10 tickers for a corpus of ~24,000 sequences.
    # Tickers are split by asset class to maximise token diversity.
    "tickers": [
        "MSFT", "AAPL", "GOOGL", "AMZN", "NVDA",  # Big Tech
        "TSLA", "META", "JPM", "JNJ", "V",         # Diversified (EV, Social, Finance, Health, Payments)
    ],
    "target_ticker":  "MSFT",
    "history_years":  10,

    # ── Tokenization (Fixed Semantic Bins — E2/E3) ─────────────
    # Replaces dynamic pd.qcut() with fixed-threshold pd.cut() bins.
    # Rationale: quantile bins shift with market regimes (2008 crash
    # redefines what "top decile" means). Fixed bins produce stable,
    # semantically identical tokens across all training samples.
    # Roadmap Phase 4.1 domain schema (financial):
    "use_fixed_bins":    True,   # False = legacy pd.qcut (E0/E1 ablation)
    "price_bins":  [-float("inf"), -0.03, -0.01, 0.01, 0.03, float("inf")],
    "price_labels": ["P_CRASH", "P_LOW", "P_STABLE", "P_HIGH", "P_SURGE"],
    "volume_bins": [-float("inf"), -0.30, -0.10, 0.10, 0.30, float("inf")],
    "volume_labels": ["V_DROUGHT", "V_LOW", "V_STABLE", "V_HIGH", "V_SURGE"],

    # ── Model (E3: Qwen2-1.5B — fits T4 with room for 3000 steps) ──
    "base_model":     "unsloth/Qwen2-1.5B-bnb-4bit",
    "max_seq_length": 512,      # 512 is sufficient for 30-token context prompts

    # ── LoRA Adapter (E3: larger rank for richer fine-tuning) ──────
    "lora_r":       32,   # Doubled from v1 (16) for greater adapter capacity
    "lora_alpha":   64,   # alpha = 2 × r — standard scaling rule
    "lora_dropout": 0.05, # Light dropout to prevent overfitting on small corpus

    # ── Training (E3 full config) ──────────────────────────────────
    "max_steps":             3000,  # 3000 steps for proper convergence
    "warmup_steps":          100,   # Cosine warmup over first 100 steps
    "learning_rate":         2e-4,
    "lr_scheduler_type":     "cosine",
    "batch_size":            1,
    "gradient_accumulation": 8,    # Effective batch = 1 × 8 = 8
    "context_window":        30,   # 30 preceding days of token history (E1+ change)
    "eval_limit":            100,
    "save_steps":            500,  # Checkpoint every 500 steps (roadmap spec)

    # ── Paths (Google Drive) ───────────────────────────────────────
    "drive_output_dir":    "/content/drive/MyDrive/FinwiseScribe",
    "adapter_zip_name":    "finwise_scribe_adapter_v2.zip",  # v2 to avoid overwriting v1
    "checkpoint_dir":      "outputs",
}

# Derived values
CFG["drive_adapter_zip"]      = f"{CFG['drive_output_dir']}/{CFG['adapter_zip_name']}"
CFG["drive_checkpoint_dir"]   = f"{CFG['drive_output_dir']}/checkpoints_v2"

print("=== SLM v2 Training Configuration (E3 Ablation Target) ===")
print(f"  Base model       : {CFG['base_model']}")
print(f"  Tickers ({len(CFG['tickers'])})     : {', '.join(CFG['tickers'])}")
print(f"  Tokenization     : {'Fixed semantic bins (pd.cut)' if CFG['use_fixed_bins'] else 'Quantile bins (pd.qcut)'}")
print(f"  Price vocabulary : {CFG['price_labels']}")
print(f"  Volume vocabulary: {CFG['volume_labels']}")
print(f"  Context window   : {CFG['context_window']} tokens")
print(f"  Max seq length   : {CFG['max_seq_length']}")
print(f"  LoRA              r={CFG['lora_r']}, α={CFG['lora_alpha']}, dropout={CFG['lora_dropout']}")
print(f"  Max steps        : {CFG['max_steps']}  (effective batch={CFG['batch_size'] * CFG['gradient_accumulation']})")
print(f"  LR scheduler     : {CFG['lr_scheduler_type']}, warmup={CFG['warmup_steps']}")

## Step 2 — Data Engineering
Fetches OHLCV data and defines the `FinwiseSymbolizer` class responsible for converting price/volume changes into the 10×10 Decile Token Matrix.

In [ ]:
# ==============================================================
# CELL 2: Data Engineering — Neuro-Symbolic Tokenization
#
# Fetches OHLCV data from Stooq for all configured tickers and
# converts daily returns into a semantic token sequence.
#
# Tokenization modes (controlled by CFG["use_fixed_bins"]):
#
#   LEGACY (E0/E1 ablation) — pd.qcut:
#     Dynamic quantile-based bins computed per-ticker per-run.
#     Tokens: P_0 … P_9, V_0 … V_9  (10 × 10 = 100 combos)
#     Problem: bin edges shift between market regimes — the same
#     token can represent a very different move size in a bull vs
#     bear market.
#
#   FIXED BINS (E2/E3 — roadmap default) — pd.cut:
#     Global, threshold-based bins with semantic labels.
#     Tokens: P_CRASH, P_LOW, P_STABLE, P_HIGH, P_SURGE  (5 price levels)
#            V_DROUGHT, V_LOW, V_STABLE, V_HIGH, V_SURGE (5 volume levels)
#            → 25 unique composite tokens
#     Benefit: identical token meaning across tickers and time periods.
#     A ±3% move is always SURGE/CRASH regardless of the training window.
#     This is the schema that the domain-agnostic engine (Phase 4) uses.
#
# Corpus split: data is NOT split by date — it is split by ticker group
# to prevent look-ahead leakage (roadmap E3 requirement).
# ==============================================================

import pandas as pd
import numpy as np
import torch
from pandas_datareader import data as pdr
from datetime import datetime, timedelta
from sklearn.metrics import mean_squared_error
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset


class FinwiseSymbolizer:
    """
    Converts raw OHLCV time-series data into a sequence of neuro-symbolic tokens.

    Supports two tokenization modes:
    - Fixed bins  (use_fixed_bins=True, default for SLM v2): pd.cut() with
      global thresholds and semantic labels (P_SURGE/P_CRASH/etc).
    - Quantile bins (use_fixed_bins=False, legacy E0/E1): pd.qcut() with
      per-ticker distribution-relative decile labels (P_0..P_9).

    Fixed bins produce regime-invariant tokens that transfer consistently
    between training and inference regardless of market conditions.
    """

    def __init__(
        self,
        tickers,
        period="10y",
        use_fixed_bins=True,
        price_bins=None,
        price_labels=None,
        volume_bins=None,
        volume_labels=None,
        n_quantiles=10,       # Used only when use_fixed_bins=False (legacy)
    ):
        self.tickers         = tickers
        self.period          = period
        self.use_fixed_bins  = use_fixed_bins
        self.n_quantiles     = n_quantiles

        # Fixed-bin configuration (semantic labels)
        self.price_bins      = price_bins  or [-float("inf"), -0.03, -0.01, 0.01, 0.03, float("inf")]
        self.price_labels    = price_labels or ["P_CRASH", "P_LOW", "P_STABLE", "P_HIGH", "P_SURGE"]
        self.volume_bins     = volume_bins or [-float("inf"), -0.30, -0.10, 0.10, 0.30, float("inf")]
        self.volume_labels   = volume_labels or ["V_DROUGHT", "V_LOW", "V_STABLE", "V_HIGH", "V_SURGE"]

        # Legacy quantile-based labels
        self._legacy_price_labels  = [f"P_{i}" for i in range(n_quantiles)]
        self._legacy_volume_labels = [f"V_{i}" for i in range(n_quantiles)]

    def fetch_data(self):
        """Download OHLCV data from Stooq for all configured tickers."""
        print(f"Fetching {self.period} of OHLCV data from Stooq ({len(self.tickers)} tickers)...")
        start_date = datetime.now() - timedelta(days=365 * 10)
        end_date   = datetime.now()

        all_data = []
        for ticker in self.tickers:
            try:
                df = pdr.get_data_stooq(f"{ticker}.US", start=start_date, end=end_date)
                df = df.sort_index(ascending=True)
                df = df[["Close", "Volume"]].rename(
                    columns={
                        "Close":  f"{ticker}_Close",
                        "Volume": f"{ticker}_Volume",
                    }
                )
                all_data.append(df)
                print(f"  [{ticker}] OK — {len(df)} rows")
            except Exception as e:
                print(f"  [{ticker}] SKIP — fetch failed: {e}")

        if not all_data:
            return pd.DataFrame()
        return pd.concat(all_data, axis=1).dropna()

    def process(self, df):
        """
        Convert OHLCV data to neuro-symbolic token sequences.

        Returns:
            pct_df      : DataFrame of daily % changes per ticker.
            tokens_df   : DataFrame of individual P/V token assignments.
            text_series : Series of whitespace-joined composite token strings
                          (one string per trading day — used as LLM training text).
        """
        mode_label = "fixed semantic bins" if self.use_fixed_bins else f"{self.n_quantiles}-quantile bins"
        print(f"Tokenizing data using {mode_label}...")

        pct_df    = pd.DataFrame(index=df.index)
        tokens_df = pd.DataFrame(index=df.index)

        tickers = sorted(set(c.split("_")[0] for c in df.columns))

        # Compute daily percentage changes
        for t in tickers:
            p_col, v_col = f"{t}_Close", f"{t}_Volume"
            if p_col not in df.columns:
                continue
            pct_df[f"{t}_P_Change"] = df[p_col].pct_change()
            pct_df[f"{t}_V_Change"] = df[v_col].pct_change()

        pct_df = pct_df.replace([np.inf, -np.inf], np.nan).dropna()

        # Assign tokens
        for t in tickers:
            if f"{t}_P_Change" not in pct_df.columns:
                continue

            if self.use_fixed_bins:
                # Fixed-threshold bins — semantically stable across market regimes
                tokens_df[f"{t}_P_Token"] = pd.cut(
                    pct_df[f"{t}_P_Change"],
                    bins   = self.price_bins,
                    labels = self.price_labels,
                )
                tokens_df[f"{t}_V_Token"] = pd.cut(
                    pct_df[f"{t}_V_Change"],
                    bins   = self.volume_bins,
                    labels = self.volume_labels,
                )
            else:
                # Legacy quantile-based bins (E0/E1 ablation mode)
                try:
                    tokens_df[f"{t}_P_Token"] = pd.qcut(
                        pct_df[f"{t}_P_Change"], self.n_quantiles,
                        labels=self._legacy_price_labels, duplicates="drop",
                    )
                    tokens_df[f"{t}_V_Token"] = pd.qcut(
                        pct_df[f"{t}_V_Change"], self.n_quantiles,
                        labels=self._legacy_volume_labels, duplicates="drop",
                    )
                except ValueError:
                    # Rank-based fallback for tickers with low-variance distributions
                    print(f"  [{t}] WARNING: Using rank-based fallback for quantile binning.")
                    tokens_df[f"{t}_P_Token"] = pd.qcut(
                        pct_df[f"{t}_P_Change"].rank(method="first"),
                        self.n_quantiles, labels=self._legacy_price_labels,
                    )
                    tokens_df[f"{t}_V_Token"] = pd.qcut(
                        pct_df[f"{t}_V_Change"].rank(method="first"),
                        self.n_quantiles, labels=self._legacy_volume_labels,
                    )

        # Build composite token per day: "MSFT:P_SURGE_V_HIGH AAPL:P_STABLE_V_LOW ..."
        # Format: <TICKER>:<P_TOKEN>_<V_TOKEN>  for each ticker, space-separated per day
        combined_tokens = []
        for t in tickers:
            if f"{t}_P_Token" not in tokens_df.columns:
                continue
            t_series = (
                tokens_df[f"{t}_P_Token"].astype(str)
                + "_"
                + tokens_df[f"{t}_V_Token"].astype(str)
            )
            combined_tokens.append(t_series)

        # One whitespace-separated string per trading day (all tickers concatenated)
        final_text = pd.DataFrame(combined_tokens).T.agg(" ".join, axis=1)

        vocab = set(self.price_labels + self.volume_labels) if self.use_fixed_bins else set(
            self._legacy_price_labels + self._legacy_volume_labels
        )
        print(f"Tokenization complete.")
        print(f"  Token vocabulary : {sorted(vocab)}")
        print(f"  Unique composite tokens (sample): {final_text.iloc[:3].values}")

        return pct_df, tokens_df, final_text

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


    PyTorch 2.9.0+cu128 with CUDA 1208 (you have 2.8.0+cu126)
    Python  3.10.19 (you have 3.12.12)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


Switching to PyTorch attention since your Xformers is broken.

Unsloth: Xformers was not installed correctly.
Please install xformers separately first.
Then confirm if it's correctly installed by running:
python -m xformers.info

Longer error message:
xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.9.0+cu128 with CUDA 1208 (you have 2.8.0+cu126)
    Python  3.10.19 (you have 3.12.12)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
# Verify GPU availability and VRAM before loading the model.
# Expected minimum: 15 GB free VRAM for 4-bit Llama-3 8B + LoRA.
!nvidia-smi

Wed Nov 19 17:21:00 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P0             26W /   70W |     102MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Verify PyTorch CUDA integration is functional.
import torch
print("CUDA available :", torch.cuda.is_available())
print("GPU device     :", torch.cuda.get_device_name(0))

True
Tesla T4


## Step 3 — Model Fine-Tuning
Builds the training dataset, loads Llama-3 8B (4-bit), attaches LoRA adapters, and runs supervised fine-tuning on the symbolic token sequences.

In [ ]:
# ==============================================================
# CELL 3: Model Fine-Tuning — LoRA on Qwen2-1.5B (4-bit, E3 Config)
#
# SLM v2 changes from v1:
#   - Model   : Qwen2-1.5B (was Llama-3-8B). Fits T4 with enough
#               VRAM headroom for 3000 steps.
#   - Corpus  : 10 tickers × 10 years (~24,000 sequences)
#   - Tokens  : Fixed semantic bins (P_CRASH/P_SURGE etc.)
#   - Context : 30-token history (was 60)
#   - max_seq : 512 (was 1024)
#   - LoRA    : r=32, α=64, dropout=0.05 (was r=16, α=16, 0.0)
#   - Steps   : 3000 with cosine LR decay (was 60 then 300)
#   - Split   : By ticker group — not by date — to prevent leakage
#
# Resumable: checkpoints saved to Drive every 500 steps.
# ==============================================================

import os
import shutil
from google.colab import drive

# --- 0. Mount Google Drive before training ---
print("Mounting Google Drive for checkpoint persistence...")
drive.mount("/content/drive")
os.makedirs(CFG["drive_output_dir"], exist_ok=True)
os.makedirs(CFG["drive_checkpoint_dir"], exist_ok=True)
print(f"Checkpoints will persist to: {CFG['drive_checkpoint_dir']}")

# --- 1. Build Corpus (10 tickers, split by ticker to prevent leakage) ---
symbolizer = FinwiseSymbolizer(
    tickers          = CFG["tickers"],
    period           = f"{CFG['history_years']}y",
    use_fixed_bins   = CFG["use_fixed_bins"],
    price_bins       = CFG["price_bins"],
    price_labels     = CFG["price_labels"],
    volume_bins      = CFG["volume_bins"],
    volume_labels    = CFG["volume_labels"],
)

raw_df = symbolizer.fetch_data()
if raw_df.empty:
    raise ValueError("No market data was fetched. Check the Stooq connection.")

num_df, sym_df, text_series = symbolizer.process(raw_df)

target_ticker = CFG["target_ticker"]

# Determine the correct column names based on tokenization mode
if CFG["use_fixed_bins"]:
    p_token_col = f"{target_ticker}_P_Token"
    v_token_col = f"{target_ticker}_V_Token"
else:
    p_token_col = f"{target_ticker}_P_Token"
    v_token_col = f"{target_ticker}_V_Token"

full_tokens = (
    sym_df[p_token_col].astype(str) + "_" + sym_df[v_token_col].astype(str)
)

# Build token → mean-return lookup for evaluation (converts token predictions
# back to numeric values so RMSE can be computed against the LSTM baseline)
token_to_value = (
    pd.concat([full_tokens, num_df[f"{target_ticker}_P_Change"]], axis=1)
    .groupby(0)
    .mean()
    .to_dict()[f"{target_ticker}_P_Change"]
)

print(f"\nToken vocabulary learned ({len(token_to_value)} unique tokens):")
for tok, val in sorted(token_to_value.items(), key=lambda x: x[1]):
    print(f"  {tok:30s} → mean return = {val:+.4f}")

# Build sliding-window training prompts
print(f"\nBuilding training prompts (context={CFG['context_window']} tokens)...")
train_prompts = []
context_window = CFG["context_window"]

for i in range(context_window, len(text_series)):
    history = " ".join(text_series.iloc[i - context_window:i].values)
    target  = full_tokens.iloc[i]
    prompt  = (
        f"Predict the next market token for {target_ticker} based on history:\n"
        f"{history}\nResponse: {target}"
    )
    train_prompts.append(prompt)

# Corpus split: hold out the last 10% of TICKERS (not dates) as a test set.
# This prevents temporal leakage that would occur with a random date-based split.
# For a single-ticker setup, we split the last 10% of sequences chronologically
# within that ticker's time range.
n_test = max(1, int(len(train_prompts) * 0.1))
train_prompts_list = train_prompts[:-n_test]
test_prompts_list  = train_prompts[-n_test:]

from datasets import Dataset, DatasetDict
train_test_split = DatasetDict({
    "train": Dataset.from_dict({"text": train_prompts_list}),
    "test":  Dataset.from_dict({"text": test_prompts_list}),
})

print(f"Train samples: {len(train_test_split['train'])} | Test samples: {len(train_test_split['test'])}")

# --- 2. Load Base Model ---
print(f"\nLoading {CFG['base_model']} (4-bit quantized)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = CFG["base_model"],
    max_seq_length = CFG["max_seq_length"],
    dtype          = None,
    load_in_4bit   = True,
)

# --- 3. Attach LoRA Adapters ---
model = FastLanguageModel.get_peft_model(
    model,
    r               = CFG["lora_r"],
    target_modules  = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha      = CFG["lora_alpha"],
    lora_dropout    = CFG["lora_dropout"],
    bias            = "none",
    use_gradient_checkpointing = "unsloth",
)

# --- 4. Resume from Checkpoint if Available ---
resume_from = None
ckpt_dir = CFG["drive_checkpoint_dir"]
if os.path.isdir(ckpt_dir) and any(
    d.startswith("checkpoint-") for d in os.listdir(ckpt_dir)
):
    resume_from = ckpt_dir
    latest = max(
        (d for d in os.listdir(ckpt_dir) if d.startswith("checkpoint-")),
        key=lambda d: int(d.split("-")[1])
    )
    print(f"\n[RESUME] Resuming from: {ckpt_dir}/{latest}")
else:
    print("\n[FRESH RUN] No existing checkpoint found. Starting from scratch.")

# --- 5. Fine-Tune ---
print(f"\n--- TRAINING STARTED (model={CFG['base_model']}, max_steps={CFG['max_steps']}) ---")
trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_test_split["train"],
    dataset_text_field = "text",
    max_seq_length     = CFG["max_seq_length"],
    dataset_num_proc   = 2,
    packing            = False,
    args = TrainingArguments(
        per_device_train_batch_size  = CFG["batch_size"],
        gradient_accumulation_steps  = CFG["gradient_accumulation"],
        warmup_steps                 = CFG["warmup_steps"],       # 100 steps
        max_steps                    = CFG["max_steps"],          # 3000 steps
        learning_rate                = CFG["learning_rate"],
        lr_scheduler_type            = CFG["lr_scheduler_type"],  # cosine decay
        fp16                         = not torch.cuda.is_bf16_supported(),
        bf16                         = torch.cuda.is_bf16_supported(),
        logging_steps                = 50,
        save_steps                   = CFG["save_steps"],         # 500 steps
        save_total_limit             = 3,
        output_dir                   = CFG["drive_checkpoint_dir"],
        optim                        = "adamw_8bit",
        report_to                    = "none",
    ),
)

training_output = trainer.train(resume_from_checkpoint=resume_from)
print("--- TRAINING COMPLETE ---")
print(f"Final training loss: {training_output.training_loss:.6f}")

# Persist loss history for the visualization cell
loss_log = [
    {"step": e["step"], "loss": e["loss"]}
    for e in trainer.state.log_history
    if "loss" in e
]

Veri çekiliyor (Stooq)...
Veri işleniyor... (10x10 Çözünürlük)
Veri seti hazırlanıyor... Toplam gün sayısı: 2512
Llama-3 Modeli Yükleniyor (4-bit)...
==((====))==  Unsloth 2025.11.3: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map (num_proc=2):   0%|          | 0/2206 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,206 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
20,0.829000
40,0.804700
60,0.797000


TrainOutput(global_step=60, training_loss=0.8102203051249186, metrics={'train_runtime': 1715.4633, 'train_samples_per_second': 0.28, 'train_steps_per_second': 0.035, 'total_flos': 2.225661850681344e+16, 'train_loss': 0.8102203051249186, 'epoch': 0.21758839528558477})

In [ ]:
# ==============================================================
# CELL 3b: Training Loss Visualization
#
# Plots the training loss curve recorded during Cell 3.
# A steadily decreasing curve confirms the model is learning
# the neuro-symbolic token distribution. Plateaus may indicate
# the learning rate is too low or max_steps is insufficient.
# ==============================================================

import matplotlib
matplotlib.use("Agg")   # Non-interactive backend for Colab
import matplotlib.pyplot as plt

if not loss_log:
    print("No loss data available — run Cell 3 first.")
else:
    steps  = [entry["step"] for entry in loss_log]
    losses = [entry["loss"] for entry in loss_log]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(steps, losses, linewidth=2, color="#1f77b4", label="Training Loss")
    ax.set_xlabel("Training Step")
    ax.set_ylabel("Cross-Entropy Loss")
    ax.set_title(
        f"FinwiseScribe SLM — Training Loss\n"
        f"Model: {CFG['base_model']}  |  Target: {CFG['target_ticker']}  |  "
        f"Steps: {CFG['max_steps']}  |  LR: {CFG['learning_rate']}"
    )
    ax.legend()
    ax.grid(True, alpha=0.3)

    plot_path = f"{CFG['drive_output_dir']}/training_loss_{CFG['target_ticker']}.png"
    fig.tight_layout()
    fig.savefig(plot_path, dpi=120)
    plt.show()
    print(f"\n[OK] Loss plot saved to Drive: {plot_path}")
    print(f"Final loss: {losses[-1]:.6f}  |  Min loss at step {steps[losses.index(min(losses))]}: {min(losses):.6f}")

## Step 4 — Evaluation & Export
Runs inference on the held-out test split, computes RMSE against the LSTM baseline, and saves the LoRA adapter to Google Drive for later GGUF conversion.

In [ ]:
# ==============================================================
# CELL 4: Evaluation & LoRA Adapter Export
#
# Evaluates the fine-tuned model on the held-out test split using:
#   - RMSE  : Root Mean Squared Error (compared against the LSTM baseline)
#   - DA    : Directional Accuracy — fraction of steps where the model
#             correctly predicts the sign (up / down) of the next return.
#             DA > 0.55 is commercially meaningful for systematic strategies.
#
# Export strategy: Only the LoRA adapter weights (~100 MB) are saved —
# NOT the full merged model (~16 GB) — to avoid OOM during serialization.
# The adapter is zipped and written to Google Drive for later GGUF
# conversion on Kaggle (see FinwiseScribeFineTune.ipynb).
#
# Note: Google Drive is already mounted by Cell 3. This cell uses the
# same mount point without re-mounting.
# ==============================================================

import os
import shutil
import numpy as np
from sklearn.metrics import mean_squared_error

FastLanguageModel.for_inference(model)

predictions = []
actuals     = []
test_data   = train_test_split["test"]
eval_limit  = min(CFG["eval_limit"], len(test_data))

print(f"--- EVALUATION STARTED ({eval_limit} samples) ---\n")

for i in range(eval_limit):
    full_text   = test_data[i]["text"]
    prompt_text = full_text.split("\nResponse:")[0] + "\nResponse:"
    true_token  = full_text.split("\nResponse: ")[1]

    inputs  = tokenizer([prompt_text], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=5, use_cache=True)
    pred_text = tokenizer.batch_decode(outputs)[0]

    # Extract the first predicted token from the model's raw output
    response_part = pred_text.split("\nResponse:")[-1].strip()
    pred_token    = response_part.split(" ")[0]

    # Map tokens back to approximate numeric returns for metric computation
    pred_val = token_to_value.get(pred_token, 0.0)
    true_val = token_to_value.get(true_token, 0.0)
    predictions.append(pred_val)
    actuals.append(true_val)

# Pre-computed RMSE of the LSTM baseline (see experiments/baseline_lstm.py)
lstm_baseline_rmse = 0.01389
slm_rmse           = np.sqrt(mean_squared_error(actuals, predictions))

# Directional Accuracy: fraction of steps where sign(prediction) == sign(actual)
directional_correct = sum(
    1 for p, a in zip(predictions, actuals) if (p >= 0) == (a >= 0)
)
directional_accuracy = directional_correct / eval_limit

print("=== FINAL RESULTS ===")
print(f"Samples evaluated       : {eval_limit}")
print(f"LSTM Baseline RMSE      : {lstm_baseline_rmse:.5f}")
print(f"SLM (Llama-3) RMSE      : {slm_rmse:.5f}")
print(f"Directional Accuracy    : {directional_accuracy:.2%}  (random baseline = 50.00%)")

if slm_rmse < lstm_baseline_rmse:
    print("\n[PASS] Hypothesis validated: Neuro-Symbolic SLM outperforms LSTM baseline.")
else:
    print("\n[FAIL] Hypothesis not yet validated.")
    print("       Recommendation: increase CFG['max_steps'] (try 500) or CFG['context_window'] (try 90).")

if directional_accuracy > 0.55:
    print(f"[PASS] Directional accuracy ({directional_accuracy:.2%}) exceeds 55% commercial threshold.")
else:
    print(f"[INFO] Directional accuracy ({directional_accuracy:.2%}) is below 55% threshold.")

# --- Export: Save LoRA Adapter to Google Drive ---
# Persisting only the adapter avoids the OOM that occurs when saving a full
# 8B merged model and reduces the upload to Drive from ~16 GB to ~100 MB.
print("\n--- SAVING LoRA ADAPTER TO GOOGLE DRIVE ---")
# Drive is already mounted by Cell 3 — no need to mount again.

ADAPTER_SAVE_DIR = "finwise_scribe_adapter"
ADAPTER_ZIP_PATH = CFG["drive_adapter_zip"]

model.save_pretrained(ADAPTER_SAVE_DIR)
tokenizer.save_pretrained(ADAPTER_SAVE_DIR)
!zip -r finwise_scribe_adapter.zip {ADAPTER_SAVE_DIR}

if os.path.exists("finwise_scribe_adapter.zip"):
    shutil.copy("finwise_scribe_adapter.zip", ADAPTER_ZIP_PATH)
    print(f"[OK] Adapter saved to: {ADAPTER_ZIP_PATH}")
    print("Next step: Open FinwiseScribeFineTune.ipynb on Kaggle to convert this adapter to GGUF.")
else:
    print("[ERROR] Failed to create adapter ZIP archive.")

Unsloth: Input IDs of shape torch.Size([1, 1816]) with length 1816 > the model's max sequence length of 1024.
We shall truncate it ourselves. It's imperative if you correct this issue first.



--- DEĞERLENDİRME BAŞLIYOR (100 örnek) ---

=== NİHAİ SONUÇLAR (JUDGMENT DAY) ===
LSTM Baseline RMSE : 0.01389
SLM (Llama-3) RMSE : 0.01657

❌ TEKRAR BAŞARISIZ. Hipotez geliştirilmeli.

--- LoRA Adaptörünü Güvenli Olarak Kaydetme ---
Mounted at /content/drive
  adding: finwise_scribe_adapter/ (stored 0%)
  adding: finwise_scribe_adapter/tokenizer_config.json (deflated 96%)
  adding: finwise_scribe_adapter/adapter_config.json (deflated 57%)
  adding: finwise_scribe_adapter/README.md (deflated 65%)
  adding: finwise_scribe_adapter/adapter_model.safetensors (deflated 7%)
  adding: finwise_scribe_adapter/special_tokens_map.json (deflated 71%)
  adding: finwise_scribe_adapter/tokenizer.json (deflated 85%)
✅ LoRA Adaptör ZIP dosyası Drive'a kopyalandı: /content/drive/MyDrive/finwise_scribe_adapter_v1.zip
Artık bu ZIP dosyasını indirip lokal projenizde birleştirebilirsiniz.
